In [1]:
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("JAX_PLATFORMS", "cpu")

def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "develop_doc").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import braincell
from braincell import Morpho
from develop_doc.neuron_diff import compare_swc_with_neuron

print("braincell version:", braincell.__version__)
print("repo_root:", repo_root)


/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.1 is installed, but it is not compatible with the installed jaxlib version 0.6.2, so it will not be used.
  warnings.warn(


braincell version: 0.0.8
repo_root: /home/swl/braincell


# `neuron_diff`: scan cached NeuroMorpho folders and compare with `NEURON`

This notebook reads the existing cache under `develop_doc/data/neuromorpho/<neuron_id>/`, finds the standardized `SWC`, and prints the `braincell` vs `neuron` report from `develop_doc.neuron_diff.compare_swc_with_neuron(...)`.


In [2]:
def load_cached_metadata(folder):
    return json.loads((Path(folder) / "metadata.json").read_text(encoding="utf-8"))


def repo_path(path_value):
    path = Path(path_value)
    return path if path.is_absolute() else repo_root / path


def find_standard_swc(folder, metadata):
    for item in metadata.get("download_items", []):
        if item.get("kind") == "standard":
            path = repo_path(item["path"])
            if path.exists():
                return path

    swc_paths = sorted(Path(folder).glob("*.swc"))
    return swc_paths[0] if swc_paths else None


In [3]:
output_root = repo_root / "develop_doc" / "data" / "neuromorpho"
neuron_metric_names = None
EXACT_MATCH_KEYS = ("n_stems", "n_branches", "n_bifurcations", "max_branch_order")
FLOAT_THRESHOLDS = {
    "total_length": 0.001,
    "total_area": 0.01,
    "total_volume": 0.01,
    "mean_radius": 0.01,
    "max_path_distance": 0.001,
    "max_euclidean_distance": 0.001,
}
SKIPPABLE_ERROR_MARKERS = (
    "not 'nonetype'",
    "requires full point geometry",
    "does not expose pt3d geometry",
    "total section length must be > 0",
    "no soma",
)


def format_metric_record(record):
    if record["available"]:
        text = f"{record['value']:.6f}"
        if record["unit"]:
            text += f" {record['unit']}"
        return text
    return f"n/a ({record['reason']})"


def has_soma_branch(tree):
    return any(branch.type == "soma" for branch in tree.branches)


def is_skippable_exception(exc):
    message = str(exc).lower()
    return any(marker in message for marker in SKIPPABLE_ERROR_MARKERS)


def unavailable_metric_names(report):
    unavailable = []
    for source_name in ("braincell", "neuron", "diff"):
        for metric_name in report["selected_metrics"]:
            record = report[source_name][metric_name]
            if not record["available"]:
                unavailable.append(f"{source_name}:{metric_name}")
    return unavailable


def compare_report_with_thresholds(report):
    metric_results = {}
    for metric_name in report["selected_metrics"]:
        braincell_record = report["braincell"][metric_name]
        neuron_record = report["neuron"][metric_name]
        diff_record = report["diff"][metric_name]

        result = {
            "braincell": None if not braincell_record["available"] else float(braincell_record["value"]),
            "neuron": None if not neuron_record["available"] else float(neuron_record["value"]),
            "abs_diff": diff_record["abs_diff"],
            "rel_diff": diff_record["rel_diff"],
            "threshold": FLOAT_THRESHOLDS.get(metric_name),
            "pass": False,
        }

        if metric_name in EXACT_MATCH_KEYS:
            result["threshold"] = None
            if braincell_record["available"] and neuron_record["available"] and diff_record["available"]:
                result["pass"] = bool(diff_record["abs_diff"] == 0.0)
            metric_results[metric_name] = result
            continue

        if not braincell_record["available"] or not neuron_record["available"] or not diff_record["available"]:
            metric_results[metric_name] = result
            continue

        rel_diff = diff_record["rel_diff"]
        if rel_diff is None:
            result["pass"] = diff_record["abs_diff"] <= 1e-12
        else:
            result["pass"] = rel_diff < FLOAT_THRESHOLDS[metric_name]
        metric_results[metric_name] = result

    integer_failed_metrics = [
        key for key in EXACT_MATCH_KEYS if key in metric_results and not metric_results[key]["pass"]
    ]
    float_failed_metrics = [
        key for key in FLOAT_THRESHOLDS if key in metric_results and not metric_results[key]["pass"]
    ]
    failed_metrics = [key for key in report["selected_metrics"] if not metric_results[key]["pass"]]
    integer_pass = not integer_failed_metrics
    float_pass = not float_failed_metrics
    return {
        "metric_results": metric_results,
        "integer_pass": integer_pass,
        "float_pass": float_pass,
        "overall_pass": integer_pass and float_pass,
        "failed_metrics": failed_metrics,
        "integer_failed_metrics": integer_failed_metrics,
        "float_failed_metrics": float_failed_metrics,
    }


if output_root.exists():
    cache_dirs = sorted(path for path in output_root.iterdir() if path.is_dir())
else:
    cache_dirs = []

neuron_runs = []
summary = {
    "scanned": 0,
    "reported": 0,
    "skipped": 0,
    "passed": 0,
    "failed": 0,
    "integer_failed": 0,
    "float_failed": 0,
    "errors": 0,
}
failed_neuron_ids = []
integer_failed_neuron_ids = []
float_failed_neuron_ids = []
error_records = []

if not cache_dirs:
    print("No cached neuron folders were found under", output_root)
else:
    for folder in cache_dirs:
        summary["scanned"] += 1
        metadata_path = folder / "metadata.json"
        swc_path = None
        neuron_id = folder.name
        if not metadata_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: missing metadata.json")
            continue

        try:
            metadata = load_cached_metadata(folder)
            neuron_id = metadata.get("neuron_id", folder.name)
        except Exception as exc:
            summary["errors"] += 1
            error_records.append((folder.name, None, str(exc)))
            print(f"[error] {folder.name}: failed to read metadata.json: {exc}")
            continue

        swc_path = find_standard_swc(folder, metadata)
        if swc_path is None or not swc_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no standardized SWC found")
            continue

        try:
            tree = Morpho.from_swc(swc_path, mode="neuron")
        except Exception as exc:
            if is_skippable_exception(exc):
                summary["skipped"] += 1
                print(f"[skip] {folder.name}: incomplete morphology in {swc_path}: {exc}")
            else:
                summary["errors"] += 1
                error_records.append((folder.name, str(swc_path), str(exc)))
                print(f"[error] {folder.name}: failed to build neuron report for {swc_path}: {exc}")
            continue

        if not has_soma_branch(tree):
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no soma branches in {swc_path}")
            continue

        try:
            neuron_result = compare_swc_with_neuron(swc_path, metric_names=neuron_metric_names)
        except Exception as exc:
            if is_skippable_exception(exc):
                summary["skipped"] += 1
                print(f"[skip] {folder.name}: incomplete metrics for {swc_path}: {exc}")
            else:
                summary["errors"] += 1
                error_records.append((folder.name, str(swc_path), str(exc)))
                print(f"[error] {folder.name}: failed to build neuron report for {swc_path}: {exc}")
            continue

        unavailable_metrics = unavailable_metric_names(neuron_result)
        if unavailable_metrics:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: unavailable metrics in {swc_path}: {unavailable_metrics}")
            continue

        comparison = compare_report_with_thresholds(neuron_result)

        summary["reported"] += 1
        if comparison["overall_pass"]:
            summary["passed"] += 1
        else:
            summary["failed"] += 1
            failed_neuron_ids.append(neuron_id)
            if not comparison["integer_pass"]:
                summary["integer_failed"] += 1
                integer_failed_neuron_ids.append(neuron_id)
            if not comparison["float_pass"]:
                summary["float_failed"] += 1
                float_failed_neuron_ids.append(neuron_id)

        neuron_runs.append(
            {
                "folder": str(folder),
                "neuron_id": metadata.get("neuron_id"),
                "neuron_name": metadata.get("neuron_name"),
                "swc_path": str(swc_path),
                "report": neuron_result,
                "comparison": comparison,
            }
        )

        print("=" * 80)
        print(f"neuron_id: {metadata.get('neuron_id')}")
        print(f"neuron_name: {metadata.get('neuron_name')}")
        print(f"folder: {folder}")
        print(f"swc_path: {swc_path}")
        print()

        for metric_name in neuron_result["selected_metrics"]:
            braincell_record = neuron_result["braincell"][metric_name]
            neuron_record = neuron_result["neuron"][metric_name]
            diff_record = neuron_result["diff"][metric_name]
            metric_result = comparison["metric_results"][metric_name]

            braincell_text = format_metric_record(braincell_record)
            neuron_text = format_metric_record(neuron_record)
            if diff_record["available"]:
                rel_text = "n/a" if diff_record["rel_diff"] is None else f"{diff_record['rel_diff']:.6%}"
                diff_text = f"abs_diff={diff_record['abs_diff']:.6f}, rel_diff={rel_text}, pass={metric_result['pass']}"
            else:
                diff_text = f"unavailable ({diff_record['reason']}), pass={metric_result['pass']}"

            print(f"{metric_name}:")
            print(f"  braincell: {braincell_text}")
            print(f"  neuron: {neuron_text}")
            print(f"  diff: {diff_text}")

        print()
        print(f"integer_pass: {comparison['integer_pass']}")
        print(f"float_pass: {comparison['float_pass']}")
        print(f"overall_pass: {comparison['overall_pass']}")
        print(f"integer_failed_metrics: {comparison['integer_failed_metrics']}")
        print(f"float_failed_metrics: {comparison['float_failed_metrics']}")
        print(f"failed_metrics: {comparison['failed_metrics']}")

    print("=" * 80)
    print("neuron report summary")
    print("scanned:", summary["scanned"])
    print("reported:", summary["reported"])
    print("skipped:", summary["skipped"])
    print("passed:", summary["passed"])
    print("failed:", summary["failed"])
    print("integer_failed:", summary["integer_failed"])
    print("float_failed:", summary["float_failed"])
    print("errors:", summary["errors"])
    print("failed neuron_ids:", failed_neuron_ids)
    print("integer-failed neuron_ids:", integer_failed_neuron_ids)
    print("float-failed neuron_ids:", float_failed_neuron_ids)
    print("error records:", error_records)

neuron_runs


--No graphics will be displayed.


neuron_id: 10069
neuron_name: Purkinje-slice-ageP43-6
folder: /home/swl/braincell/develop_doc/data/neuromorpho/10069
swc_path: /home/swl/braincell/develop_doc/data/neuromorpho/10069/Purkinje-slice-ageP43-6.CNG.swc

total_length:
  braincell: 6443.987206 um
  neuron: 6443.987063 um
  diff: abs_diff=0.000143, rel_diff=0.000002%, pass=True
total_area:
  braincell: 27770.069859 um^2
  neuron: 27770.069365 um^2
  diff: abs_diff=0.000494, rel_diff=0.000002%, pass=True
total_volume:
  braincell: 14335.991608 um^3
  neuron: 14335.991457 um^3
  diff: abs_diff=0.000151, rel_diff=0.000001%, pass=True
mean_radius:
  braincell: 0.680084 um
  neuron: 0.680084 um
  diff: abs_diff=0.000000, rel_diff=0.000000%, pass=True
n_branches:
  braincell: 758.000000 count
  neuron: 758.000000 count
  diff: abs_diff=0.000000, rel_diff=0.000000%, pass=True
n_stems:
  braincell: 1.000000 count
  neuron: 1.000000 count
  diff: abs_diff=0.000000, rel_diff=0.000000%, pass=True
n_bifurcations:
  braincell: 378.000000 c

[{'folder': '/home/swl/braincell/develop_doc/data/neuromorpho/10069',
  'neuron_id': 10069,
  'neuron_name': 'Purkinje-slice-ageP43-6',
  'swc_path': '/home/swl/braincell/develop_doc/data/neuromorpho/10069/Purkinje-slice-ageP43-6.CNG.swc',
  'report': {'path': '/home/swl/braincell/develop_doc/data/neuromorpho/10069/Purkinje-slice-ageP43-6.CNG.swc',
   'selected_metrics': ('total_length',
    'total_area',
    'total_volume',
    'mean_radius',
    'n_branches',
    'n_stems',
    'n_bifurcations',
    'max_branch_order',
    'max_path_distance',
    'max_euclidean_distance'),
   'braincell': {'total_length': {'available': True,
     'value': 6443.987205922132,
     'unit': 'um'},
    'total_area': {'available': True,
     'value': 27770.069858870564,
     'unit': 'um^2'},
    'total_volume': {'available': True,
     'value': 14335.991607519407,
     'unit': 'um^3'},
    'mean_radius': {'available': True,
     'value': 0.6800836251241766,
     'unit': 'um'},
    'n_branches': {'availabl